In [ ]:
import irw
import numpy as np
import pandas as pd

In [ ]:
gilbert = irw.fetch("gilbert_meta_2")

In [ ]:
response_dataframe = gilbert.pivot(index = "id", columns = "item", values = "resp")
x_sq = response_dataframe.to_numpy(dtype = float)
print(x_sq)

In [ ]:
no_items, no_students = gilbert["item"].nunique(), gilbert["id"].nunique()

In [ ]:
# defining rasch probability function (P{x_sq = 1})
# using alternative form of function for sigma for better computational compatibility with asymptotic values
def sigma(z):
    return 1 / (1 + np.exp(-z))

def p_correct(a_s,d_q):
    return sigma(a_s - d_q)

p_correct(0,0)

In [ ]:
# Finding total student scores and total item scores (sufficient statistics)
A_s = x_sq.sum(axis = 1) # Sum along the rows
print(A_s)
print(A_s.shape)
D_q = x_sq.sum(axis = 0) # Sum along the columns
print(D_q)
print(D_q.shape)

# finding the indexes of people who got full or no marks to be able to check their parameters later
for i, n in enumerate(A_s): # Trying out the enumerate fn
    if n == 20 or n == 0:
        print(n, i, sep = ": ")


In [ ]:
# optimised log likelihood function
def log_likelihood(x_sq, a_s, d_q):

    a_minus_d = a_s[:,None] - d_q[None,:]
    answer = ((x_sq*a_minus_d) - (np.logaddexp(0,a_minus_d))).sum()

    return answer


In [ ]:
# 'bisection' code adjusted for this project

def maximising_f(f, xmin, xmax, maxiter = 50, tol = 1e-8):

    a = xmin
    b = xmax

    for _ in range(maxiter):

        m = (a + b)/2
        l = (a + m)/2
        r = (b + m)/2

        vals = np.array([f(a), f(b), f(m), f(l), f(r)])

        max_val = np.argmax(vals)

        if max_val == 0 or max_val == 3:
            b = m 
        elif max_val == 1 or max_val == 4:
            a = m
        else:
            a = l
            b = r 
        
        if abs(a - b) < tol:
            break
        
        
    return (a + b)/2

In [ ]:
# gradient 'ascent' code
a_s = np.zeros(no_students)
d_q = np.zeros(no_items)
gamma_a = None
gamma_d = None
gamma_converged = False 

tol = 1e-5
gamma_a_tol = 1e-2
gamma_d_tol = 1e-3
max_iterations = 10000

log_likelihoods = []

for i in range(max_iterations):

    P_sq = sigma(a_s[:,None] - d_q[None,:])

    expected_A_s = P_sq.sum(axis = 1)
    expected_D_q = P_sq.sum(axis = 0)

    dl_da = A_s - expected_A_s
    dl_dd = expected_D_q - D_q

    if not gamma_converged:
        likelihood_a = lambda n: log_likelihood(x_sq, a_s + (n*dl_da), d_q)
        likelihood_d = lambda n: log_likelihood(x_sq, a_s, d_q + (n*dl_dd))

        new_gamma_a = maximising_f(likelihood_a, 0, 0.5)
        new_gamma_d = maximising_f(likelihood_d, 0, 0.5)

        if gamma_a is not None and gamma_d is not None:

            if abs(new_gamma_a - gamma_a) < gamma_a_tol and abs(new_gamma_d - gamma_d) < gamma_d_tol:
                gamma_converged = True
                print(f"iteration {i}, Both gammas converged")
                print(f"Gamma a = {new_gamma_a}, Gamma b = {new_gamma_d}")

    gamma_a = new_gamma_a
    gamma_d = new_gamma_d
    
    a_s += gamma_a * dl_da
    d_q += gamma_d * dl_dd
    
    mean_d = d_q.mean()
    d_q -= mean_d
    a_s -= mean_d
        
    value = log_likelihood(x_sq, a_s, d_q)
    log_likelihoods.append(value)

    if i != 0 and abs(value - log_likelihoods[-2]) < tol:
        final_iteration = i
        print()
        print("Converged!")
        print("Convergence details:")
        print(f"iteration: {final_iteration}")
        print(f"max likelihood value: {value:.2f}")
        break


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
x = np.arange(final_iteration + 1)
print(x.shape)
print(len(log_likelihoods))

plt.plot(x, log_likelihoods, c = "m")
plt.title("Log-likelihood over Iterations of Gradient Ascent")
plt.xlabel("Iteration Number")
plt.ylabel("Log-Likelihood Value")
plt.show()

In [ ]:
print(a_s)
print(d_q)

In [ ]:
item_p = gilbert.groupby("item")["resp"].mean()

item_table = pd.DataFrame({
    "Observed proportion correct": item_p.to_numpy(),
    "Estimated difficulty": d_q
}, index = item_p.index)

item_table.index.name = "Item"
item_table = item_table.round(3)

display(item_table)

In [ ]:
total_scores = np.unique(A_s).astype(int)

ability_table = pd.DataFrame({
    "Total score": total_scores,
    "Estimated ability": [
        a_s[A_s == score].mean()
        for score in total_scores
    ],
    "Number of students": [
        np.sum(A_s == score)
        for score in total_scores
    ]
})

ability_table = ability_table.set_index("Total score")
ability_table.index.name = "Total score"
ability_table = ability_table.round({"Estimated ability": 3})

display(ability_table)